# Resume Suitability Classification (Phase 5)

In notebook 03 we built the hybrid resume-job matcher and ended up with a label on every
candidate-job pair: **Highly Suitable / Suitable / Not Suitable**. This notebook trains models to
predict that label from features, instead of relying on the hand-tuned scoring rule.

Two things worth being upfront about before any code:

- This is a **supervised** multi-class problem. We already have labels, and every metric we were
  asked for (accuracy, precision, recall, F1, confusion matrix) needs ground-truth labels - so
  clustering / unsupervised learning isn't the right fit here.
- The label came from `final_match_score = 0.7*text_sim + 0.3*skill_overlap` with fixed cut-offs.
  If I feed the models `final_match_score` (or its two ingredients) they'll just rediscover the
  threshold and score ~100%. That's leakage, not learning. Those columns are dropped on purpose;
  the models have to work off the actual resume / job text plus a few engineered features.

Order of attack: classical models first (Logistic Regression, Linear SVM, Random Forest, XGBoost),
then check whether a small neural net and sentence-transformer embeddings actually earn their keep.

In [ ]:
import re
import json
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd()))
from project_paths import CLEANED_DIR

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

SEED = 42
np.random.seed(SEED)

DATA = CLEANED_DIR
MODELS_DIR = DATA / "models"
REPORTS_DIR = DATA / "reports"
MODELS_DIR.mkdir(parents=True, exist_ok=True)


LABELS = ["Not Suitable", "Suitable", "Highly Suitable"]

In [ ]:
pairs = pd.read_csv(DATA / "matching" / "match_scores.csv")

resume = pd.read_csv(
    DATA / "master" / "resume_master.csv",
    usecols=["candidate_id", "resume_clean", "skill_count", "years_experience",
             "highest_education", "resume_word_count", "resume_completeness_score"],
)
jobs = pd.read_csv(
    DATA / "master" / "job_master.csv",
    usecols=["job_id", "description_clean", "required_skill_count", "description_word_count"],
)

df = (pairs
      .merge(resume, on="candidate_id", how="left")
      .merge(jobs, on="job_id", how="left"))

print(df.shape)
df[["candidate_id", "job_id", "candidate_role", "job_title", "suitability"]].head()

## A quick look at what we're working with

24,810 pairs, but only ~2,481 distinct resumes (each candidate was matched to their top-10 jobs).
That detail matters later for how we split train/test.

In [ ]:
print(df["suitability"].value_counts(), "\n")
print((df["suitability"].value_counts(normalize=True) * 100).round(1))

ax = sns.countplot(data=df, x="suitability", order=LABELS, palette="viridis")
ax.set_title("Label distribution")
ax.set_xlabel("")
for c in ax.containers:
    ax.bar_label(c)
plt.show()

So the classes are imbalanced - *Highly Suitable* is the minority at ~13%. I'll keep this in mind:
plain accuracy will look flattering, so model selection is done on **macro-F1**, and every model gets
class weighting so the rare class isn't ignored.

In [ ]:
# how complete are the features we plan to lean on?
feat_peek = ["skill_count", "years_experience", "highest_education", "resume_word_count",
             "resume_completeness_score", "required_skill_count", "description_word_count"]
df[feat_peek].isna().mean().round(3).sort_values(ascending=False)

`years_experience` is missing for ~60% of resumes and `highest_education` for ~28%. That's too much to
drop, so I'll impute (median for experience, an "unknown" rank for education) and add a flag for whether
experience was actually known.

## Feature engineering

What goes in:
- **Text**: resume + job description, turned into TF-IDF then squeezed down with TruncatedSVD. This is
  where the real signal lives - it forces the models to actually read the content.
- **Engineered numerics**: skill match/miss counts, resume skill count, experience, education rank,
  word counts, completeness.

What stays out (deliberately): `final_match_score`, `text_match_pct`, `skill_overlap_pct`, `rank`, and
all the id/name columns. Those either leak the label or carry no signal.

In [ ]:
def count_pipe(s):
    # the matcher stored skill lists as "a|b|c" strings
    if not isinstance(s, str) or not s.strip():
        return 0
    return len([x for x in s.split("|") if x])

df["n_matching_skills"] = df["matching_skills"].map(count_pipe)
df["n_missing_skills"] = df["missing_skills"].map(count_pipe)

# education is ordinal -> rank it; anything unknown sits at 0
edu_rank = {"high_school": 1, "diploma": 2, "associate": 3,
            "bachelor": 4, "master": 5, "doctorate": 6}
df["education_level"] = df["highest_education"].map(edu_rank).fillna(0)

df["has_experience_info"] = df["years_experience"].notna().astype(int)
df["years_experience"] = df["years_experience"].fillna(df["years_experience"].median())

df["combined_text"] = df["resume_clean"].fillna("") + " " + df["description_clean"].fillna("")

NUM_FEATURES = ["n_matching_skills", "n_missing_skills", "skill_count", "years_experience",
                "has_experience_info", "education_level", "resume_word_count",
                "resume_completeness_score", "required_skill_count", "description_word_count"]

df[NUM_FEATURES] = df[NUM_FEATURES].fillna(0)
df[NUM_FEATURES].describe().T[["mean", "std", "min", "max"]].round(2)

## Train / test split - grouped by candidate

The obvious mistake here would be a random row split. Because each candidate appears in 10 rows that
all share the *same resume text*, a random split would put a candidate's resume on both sides and the
models would partly memorise resumes rather than generalise. So I split by `candidate_id` with
`GroupShuffleSplit` - a resume is fully in train or fully in test, never both.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["candidate_id"].to_numpy()
y_all = df["suitability"].to_numpy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(gss.split(df, y_all, groups))

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

assert not (set(train_df["candidate_id"]) & set(test_df["candidate_id"])), "candidate leaked across split!"
print(f"train {len(train_df):,} rows  |  test {len(test_df):,} rows")
print("train mix:", train_df["suitability"].value_counts(normalize=True).round(3).to_dict())
print("test  mix:", test_df["suitability"].value_counts(normalize=True).round(3).to_dict())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

# everything below is fit on TRAIN ONLY, then applied to test - no peeking
tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2),
                        min_df=5, max_df=0.85, max_features=20000, sublinear_tf=True)
svd = TruncatedSVD(n_components=120, random_state=SEED)
scaler = StandardScaler()

Xtr_txt = svd.fit_transform(tfidf.fit_transform(train_df["combined_text"]))
Xte_txt = svd.transform(tfidf.transform(test_df["combined_text"]))
print("variance kept by SVD:", round(svd.explained_variance_ratio_.sum(), 3))

Xtr_num = scaler.fit_transform(train_df[NUM_FEATURES])
Xte_num = scaler.transform(test_df[NUM_FEATURES])

X_train = np.hstack([Xtr_txt, Xtr_num]).astype(np.float32)
X_test = np.hstack([Xte_txt, Xte_num]).astype(np.float32)
y_train = train_df["suitability"].to_numpy()
y_test = test_df["suitability"].to_numpy()
print("feature matrix:", X_train.shape)

## Part A - Classical models

Logistic Regression, Linear SVM, Random Forest and XGBoost. All four:
- use class weighting (or balanced sample weights) for the imbalance,
- are tuned on **macro-F1** with grouped cross-validation (so the CV folds also respect the
  candidate grouping),
- are scored on the held-out test set at the end.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix, f1_score)

cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
train_groups = train_df["candidate_id"].to_numpy()

results = {}  

def evaluate(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    results[name] = {"accuracy": acc, "precision_macro": p, "recall_macro": r,
                     "f1_macro": f, "y_pred": y_pred}
    print(f"{name:<22} acc={acc:.3f}  macroP={p:.3f}  macroR={r:.3f}  macroF1={f:.3f}")
    return results[name]

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = GridSearchCV(
    LogisticRegression(max_iter=2000, class_weight="balanced"),
    {"C": [0.3, 1.0, 3.0]},
    scoring="f1_macro", cv=cv, n_jobs=-1,
)
logreg.fit(X_train, y_train, groups=train_groups)
print("best C =", logreg.best_params_["C"], "| cv macro-F1 =", round(logreg.best_score_, 3))
evaluate("Logistic Regression", y_test, logreg.predict(X_test));

In [ ]:
from sklearn.svm import LinearSVC

svm = GridSearchCV(
    LinearSVC(class_weight="balanced"),
    {"C": [0.1, 0.5, 1.0]},
    scoring="f1_macro", cv=cv, n_jobs=-1,
)
svm.fit(X_train, y_train, groups=train_groups)
print("best C =", svm.best_params_["C"], "| cv macro-F1 =", round(svm.best_score_, 3))
evaluate("Linear SVM", y_test, svm.predict(X_test));

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# first pass with unconstrained trees gave train F1 ~1.0 vs test ~0.87 - classic memorisation.
# so the grid here is really the overfitting knob: cap depth, force a minimum leaf size, and only
# consider a sqrt of the features per split. costs a little train accuracy, shrinks the gap a lot.
rf = GridSearchCV(
    RandomForestClassifier(class_weight="balanced_subsample", max_features="sqrt",
                           random_state=SEED, n_jobs=-1),
    {"n_estimators": [400], "max_depth": [10, 16], "min_samples_leaf": [5, 12]},
    scoring="f1_macro", cv=cv, n_jobs=-1,
)
rf.fit(X_train, y_train, groups=train_groups)
print("best params:", rf.best_params_, "| cv macro-F1 =", round(rf.best_score_, 3))
evaluate("Random Forest", y_test, rf.predict(X_test));

In [ ]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight


label_to_int = {lab: i for i, lab in enumerate(LABELS)}
ytr_int = np.array([label_to_int[v] for v in y_train])
yte_int = np.array([label_to_int[v] for v in y_test])
sample_w = compute_sample_weight("balanced", ytr_int)


xgb = XGBClassifier(
    n_estimators=350, max_depth=4, learning_rate=0.07,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=5, gamma=0.5, reg_lambda=3.0,
    objective="multi:softprob", num_class=3, eval_metric="mlogloss",
    random_state=SEED, n_jobs=-1, tree_method="hist",
)
xgb.fit(X_train, ytr_int, sample_weight=sample_w)
xgb_pred = np.array(LABELS)[xgb.predict(X_test)]
evaluate("XGBoost", y_test, xgb_pred);

In [ ]:
def gap(name, model, integer=False):
    if integer:
        tr = f1_score(ytr_int, model.predict(X_train), average="macro")
        te = f1_score(yte_int, model.predict(X_test), average="macro")
    else:
        tr = f1_score(y_train, model.predict(X_train), average="macro")
        te = f1_score(y_test, model.predict(X_test), average="macro")
    print(f"{name:<22} train={tr:.3f}  test={te:.3f}  gap={tr - te:+.3f}")

gap("Logistic Regression", logreg)
gap("Linear SVM", svm)
gap("Random Forest", rf)
gap("XGBoost", xgb, integer=True)

In [ ]:
from sklearn.model_selection import learning_curve

sizes, tr_sc, va_sc = learning_curve(
    LogisticRegression(max_iter=2000, class_weight="balanced", C=logreg.best_params_["C"]),
    X_train, y_train, groups=train_groups, cv=cv,
    scoring="f1_macro", train_sizes=np.linspace(0.2, 1.0, 5), n_jobs=-1,
)
plt.plot(sizes, tr_sc.mean(1), "o-", label="train")
plt.plot(sizes, va_sc.mean(1), "o-", label="validation")
plt.xlabel("training examples"); plt.ylabel("macro-F1")
plt.title("Learning curve - Logistic Regression"); plt.legend()
plt.show()

## Part B - Advanced models

Now the optional bit: does a small neural network or sentence-transformer embeddings beat the classical
baselines? Installing torch + sentence-transformers (skip if already present).

In [ ]:
%pip install -q torch sentence-transformers

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)

### Simple neural network (PyTorch)

A small MLP on the **same engineered feature matrix** the classical models used - that keeps the
comparison honest. The overfitting guards are dropout, weight decay, and early stopping on a
validation slice (again carved out by candidate). Loss is class-weighted for the imbalance.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit


inner = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
tr_i, va_i = next(inner.split(X_train, y_train, train_groups))

def t_float(a): return torch.tensor(np.asarray(a), dtype=torch.float32)
def t_long(a):  return torch.tensor(np.asarray(a), dtype=torch.long)

def class_weights(y_int, n=3):
    c = np.bincount(y_int, minlength=n)
    return torch.tensor(c.sum() / (n * c), dtype=torch.float32)

class MLP(nn.Module):
    def __init__(self, d_in, n_cls=3, p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 128), nn.ReLU(), nn.Dropout(p),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(p),
            nn.Linear(64, n_cls),
        )
    def forward(self, x):
        return self.net(x)

def train_mlp(Xtr, ytr, Xva, yva, weight, max_epochs=100, patience=8):
    model = MLP(Xtr.shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss(weight=weight.to(device))
    loader = DataLoader(TensorDataset(Xtr.to(device), ytr.to(device)), batch_size=256, shuffle=True)
    best, best_state, wait, hist = 1e9, None, 0, {"train": [], "val": []}
    for _ in range(max_epochs):
        model.train()
        running = 0.0
        for xb, yb in loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward(); opt.step()
            running += loss.item() * len(xb)
        model.eval()
        with torch.no_grad():
            vloss = loss_fn(model(Xva.to(device)), yva.to(device)).item()
        hist["train"].append(running / len(Xtr)); hist["val"].append(vloss)
        if vloss < best - 1e-4:
            best, best_state, wait = vloss, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    return model, hist

In [ ]:
w = class_weights(ytr_int[tr_i])
Xtr_t, Xva_t = t_float(X_train[tr_i]), t_float(X_train[va_i])
ytr_t, yva_t = t_long(ytr_int[tr_i]), t_long(ytr_int[va_i])

mlp_feat, hist = train_mlp(Xtr_t, ytr_t, Xva_t, yva_t, w)

plt.plot(hist["train"], label="train")
plt.plot(hist["val"], label="val")
plt.xlabel("epoch"); plt.ylabel("cross-entropy")
plt.title("Neural net training (engineered features)"); plt.legend()
plt.show()

mlp_feat.eval()
with torch.no_grad():
    pred = mlp_feat(t_float(X_test).to(device)).argmax(1).cpu().numpy()
evaluate("Neural Net (features)", y_test, np.array(LABELS)[pred]);

### Sentence-Transformer embeddings (all-MiniLM-L6-v2)

In [ ]:
from sentence_transformers import SentenceTransformer

emb_cache = MODELS_DIR / "sbert_embeddings.npz"

if emb_cache.exists():
    z = np.load(emb_cache, allow_pickle=True)
    res_emb, job_emb = z["res"].item(), z["job"].item()
    print("loaded cached embeddings")
else:
    encoder = SentenceTransformer("all-MiniLM-L6-v2", device=device)
    res_text = df.drop_duplicates("candidate_id").set_index("candidate_id")["resume_clean"].fillna("").to_dict()
    job_text = df.drop_duplicates("job_id").set_index("job_id")["description_clean"].fillna("").to_dict()

    def encode(text_by_id):
        ids = list(text_by_id)
        vecs = encoder.encode([text_by_id[i] for i in ids], batch_size=128,
                              normalize_embeddings=True, show_progress_bar=True)
        return {i: v for i, v in zip(ids, vecs)}

    res_emb, job_emb = encode(res_text), encode(job_text)
    np.savez(emb_cache, res=res_emb, job=job_emb)
    print(f"encoded {len(res_emb)} resumes and {len(job_emb)} jobs")

In [ ]:
def stack_emb(frame):
    r = np.vstack([res_emb[c] for c in frame["candidate_id"]])
    j = np.vstack([job_emb[k] for k in frame["job_id"]])
    return np.hstack([r, j]).astype(np.float32)

Xtr_sb = stack_emb(train_df)
Xte_sb = stack_emb(test_df)
print("SBERT feature matrix:", Xtr_sb.shape)

# reuse the same MLP + the same grouped validation indices on the embedding features
mlp_sb, hist_sb = train_mlp(t_float(Xtr_sb[tr_i]), ytr_t, t_float(Xtr_sb[va_i]), yva_t, w)

mlp_sb.eval()
with torch.no_grad():
    pred = mlp_sb(t_float(Xte_sb).to(device)).argmax(1).cpu().numpy()
evaluate("SBERT + Neural Net", y_test, np.array(LABELS)[pred]);

## Results

In [ ]:
metric_cols = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]
summary = pd.DataFrame({k: {m: v[m] for m in metric_cols} for k, v in results.items()}).T
summary = summary.sort_values("f1_macro", ascending=False).round(3)
summary

In [ ]:
ax = summary["f1_macro"].plot(kind="barh", color="steelblue")
ax.invert_yaxis()
ax.set_xlabel("macro-F1 (held-out test)")
ax.set_title("Model comparison")
for i, v in enumerate(summary["f1_macro"]):
    ax.text(v + 0.004, i, f"{v:.3f}", va="center")
plt.show()

In [ ]:
best_name = summary.index[0]
print("Best model by macro-F1:", best_name, "\n")
print(classification_report(y_test, results[best_name]["y_pred"], labels=LABELS, digits=3))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (name, res) in zip(axes.ravel(), results.items()):
    cm = confusion_matrix(y_test, res["y_pred"], labels=LABELS)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=LABELS, yticklabels=LABELS, ax=ax)
    ax.set_title(f"{name}  (F1={res['f1_macro']:.3f})", fontsize=10)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
for ax in axes.ravel()[len(results):]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "ml_confusion_matrices.png", dpi=120, bbox_inches="tight")
plt.show()

### What the models lean on

Feature importances from the Random Forest. I'm only plotting the engineered numerics (the 120 SVD
text dimensions aren't individually interpretable) just to see how much the structured signal matters
next to the text.

In [ ]:
feat_names = [f"svd_{i}" for i in range(120)] + NUM_FEATURES
imp = pd.Series(rf.best_estimator_.feature_importances_, index=feat_names)
imp[NUM_FEATURES].sort_values().plot(kind="barh", color="darkseagreen",
                                     title="Random Forest - engineered feature importance")
plt.xlabel("importance"); plt.tight_layout()
plt.show()

## Save the winner and the metrics

Persist the best model, the fitted feature pipeline (so a new resume/job pair can be transformed the
same way at inference), and a metrics report for the rest of the project to reference.

In [ ]:
import joblib

estimators = {
    "Logistic Regression": logreg.best_estimator_,
    "Linear SVM": svm.best_estimator_,
    "Random Forest": rf.best_estimator_,
    "XGBoost": xgb,
}

if best_name in estimators:
    joblib.dump(estimators[best_name], MODELS_DIR / "best_model.joblib")
    joblib.dump({"tfidf": tfidf, "svd": svd, "scaler": scaler,
                 "num_features": NUM_FEATURES, "labels": LABELS},
                MODELS_DIR / "feature_pipeline.joblib")
    saved = "best_model.joblib + feature_pipeline.joblib"
else:
    net = mlp_feat if best_name == "Neural Net (features)" else mlp_sb
    torch.save(net.state_dict(), MODELS_DIR / "best_model_nn.pt")
    saved = "best_model_nn.pt"

report = {
    "task": "resume_suitability_classification",
    "n_pairs": int(len(df)),
    "split": "GroupShuffleSplit by candidate_id, 80/20",
    "labels": LABELS,
    "best_model": best_name,
    "metrics": {name: {m: round(float(v[m]), 4) for m in metric_cols}
                for name, v in results.items()},
}
with open(REPORTS_DIR / "ml_classification_metrics.json", "w") as fh:
    json.dump(report, fh, indent=2)

print("saved:", saved)
print(json.dumps(report, indent=2))